In [1]:
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### Read and Chunk Large PDF

In [1]:
# Load large PDF
reader = PdfReader("C:\\genai\\dataset\\what-is-a-data-lake-evolution-of-data-lakes-in-the-age-of-cloud-computing.pdf")
raw_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
docs = splitter.create_documents([raw_text])
print(f"📄 Total Chunks: {len(docs)}")


NameError: name 'PdfReader' is not defined

### Store in Chroma Vector DB and presist directory

In [3]:

# Free embedding model
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in ChromaDB
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./chroma_large_pdf"
)
vectordb.persist()


C:\Users\ajsin\AppData\Local\Temp\ipykernel_27828\907654698.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\ajsin\AppData\Local\Temp\ipykernel_27828\907654698.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


### Query & Retrieve Relevant Chunks

In [4]:
retriever = vectordb.as_retriever(search_kwargs={"k": 5})   ##Converts it to a vector embedding using the same model used to store documents,
query = "Summarize the full document"
results = retriever.get_relevant_documents(query)

context = "\n".join([doc.page_content for doc in results])


C:\Users\ajsin\AppData\Local\Temp\ipykernel_27828\3718061173.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(query)


In [5]:


model_id = "google/long-t5-tglobal-base"
# model_id= "meta-llama/Llama-3.3-70B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Format the prompt
prompt = "summarize: " + context

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=16384)

# Generate summary
outputs = model.generate(inputs.input_ids, max_length=512, num_beams=4)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)




### Format into Bullets (if summary has multiple sentences)

In [6]:
import textwrap

def print_bullet_summary(text, width=100):
    sentences = text.split(". ")
    for sentence in sentences:
        if sentence.strip():
            wrapped = textwrap.fill("• " + sentence.strip(), width=width)
            print(wrapped + "\n")



### print summarization text

In [7]:
# Example usage
print_bullet_summary(summary)

• I’ d also like to thank the reviewers—Vinoth Chandar, Jobinesh Purushothaman, Aaditya Maruthi, and
Shailesh Deshpande—and editor Melissa Potter for their feedback and input, which made a huge ing
without limitation responsibility for damages resulting from the use of or reli ance on this work

• If any code samples or other technology this work contains or describes is subject to open source
licenses or the intellectual property rights of oth ers, it is your responsibility to ensure that
your use thereof complies with such licen ses and/or rights.

